# 엑셀/PDF 파일 읽기 실습

- `pandas`, `numpy`를 사용하지 않고 엑셀과 PDF 파일을 직접 읽어보는 예제
- 엑셀 파일은 `openpyxl`, PDF 파일은 `pypdf` 패키지를 사용
- 일반 텍스트 파일처럼 `open(...).read()`만으로 읽기 어려운 구조화 파일을 다루는 흐름을 확인한다.


## 패키지 설치

터미널에서 아래 명령어를 한 번 실행한다.

```bash
python -m pip install openpyxl pypdf
```

> 실습 파일인 `students.xlsx`, `io_sample.pdf`는 현재 노트북과 같은 폴더에 미리 준비되어 있다.


In [1]:
# 예제: 실습 파일 경로와 존재 여부 확인
# pathlib.Path는 파일/폴더 경로를 문자열보다 편하게 다루기 위한 클래스이다.
# 현재 노트북이 실행되는 위치를 기준으로 students.xlsx 파일을 가리킨다.
from pathlib import Path

# 엑셀 실습 파일 경로를 Path 객체로 생성한다.
excel_path = Path('students.xlsx')

# PDF 실습 파일 경로를 Path 객체로 생성한다.
pdf_path = Path('io_sample.pdf')

# 어떤 파일을 읽으려고 하는지 경로를 먼저 출력해서 확인한다.
print('excel_path:', excel_path)

# exists()는 해당 경로에 파일 또는 폴더가 실제로 존재하는지 True/False로 반환한다.
print('excel_path.exists():', excel_path.exists())

# PDF 파일 경로도 같은 방식으로 확인한다.
print('pdf_path:', pdf_path)
print('pdf_path.exists():', pdf_path.exists())


excel_path: students.xlsx
excel_path.exists(): True
pdf_path: io_sample.pdf
pdf_path.exists(): True


## 엑셀 파일 읽기

엑셀 파일은 단순 문자열 파일이 아니라 여러 시트, 셀, 서식 정보가 들어 있는 구조화 파일이다. 그래서 `openpyxl` 같은 전용 라이브러리로 읽는다.


In [2]:
# 예제: 엑셀 파일을 열고 시트 정보 확인
# openpyxl은 pandas 없이 엑셀 파일(.xlsx)을 읽고 쓸 수 있게 해주는 라이브러리이다.
from openpyxl import load_workbook

# load_workbook()은 엑셀 파일을 열어 Workbook 객체로 반환한다.
# data_only=True는 셀에 수식이 있는 경우 수식 자체가 아니라 계산된 값을 읽겠다는 의미이다.
workbook = load_workbook(excel_path, data_only=True)

# sheetnames는 엑셀 파일 안에 들어 있는 시트 이름 목록이다.
print('workbook.sheetnames:', workbook.sheetnames)

# students라는 이름의 시트를 선택한다.
# 엑셀 하단 탭에서 특정 시트를 클릭하는 것과 비슷하다.
worksheet = workbook['students']

# title은 현재 선택한 시트의 이름이다.
print('worksheet.title:', worksheet.title)

# max_row는 데이터가 들어 있는 마지막 행 번호이다.
# 예제 파일은 헤더 1행 + 학생 데이터 4행이므로 5가 출력된다.
print('worksheet.max_row:', worksheet.max_row)

# max_column은 데이터가 들어 있는 마지막 열 번호이다.
# id, name, course, score, passed 총 5개 열이므로 5가 출력된다.
print('worksheet.max_column:', worksheet.max_column)


workbook.sheetnames: ['students']
worksheet.title: students
worksheet.max_row: 5
worksheet.max_column: 5


In [3]:
# 예제: 엑셀의 특정 셀 값 읽기
# 엑셀 셀 주소 방식으로 값을 읽을 수 있다.
# A1은 첫 번째 열 A, 첫 번째 행 1을 의미하며 여기에는 컬럼명이 들어 있다.
print("worksheet['A1'].value:", worksheet['A1'].value)

# B2는 두 번째 열 B, 두 번째 행 2를 의미한다.
# 예제 파일에서는 첫 번째 학생의 이름이 들어 있다.
print("worksheet['B2'].value:", worksheet['B2'].value)

# cell(row=행번호, column=열번호) 방식으로도 셀에 접근할 수 있다.
# Python 인덱스와 달리 openpyxl의 행/열 번호는 1부터 시작한다.
print('worksheet.cell(row=2, column=4).value:', worksheet.cell(row=2, column=4).value)


worksheet['A1'].value: id
worksheet['B2'].value: 홍길동
worksheet.cell(row=2, column=4).value: 92


In [4]:
# 예제: 엑셀의 모든 행을 한 줄씩 읽기
# iter_rows()는 워크시트의 행을 위에서 아래로 하나씩 반복한다.
# values_only=True를 사용하면 Cell 객체가 아니라 셀 안의 실제 값만 tuple로 가져온다.
for row in worksheet.iter_rows(values_only=True):
    # row에는 엑셀의 한 행 데이터가 tuple 형태로 들어 있다.
    print('row:', row)


row: ('id', 'name', 'course', 'score', 'passed')
row: (1, '홍길동', 'Python IO', 92, True)
row: (2, '이순신', 'Python IO', 85, True)
row: (3, '유관순', 'Python IO', 78, True)
row: (4, '신사임당', 'Python IO', 64, False)


In [5]:
# 예제: 엑셀 데이터를 dict 리스트로 변환
# iter_rows() 결과를 list로 변환하면 엑셀 전체 데이터를 행 단위로 보관할 수 있다.
rows = list(worksheet.iter_rows(values_only=True))

# 첫 번째 행은 실제 데이터가 아니라 컬럼명이다.
# 예: ('id', 'name', 'course', 'score', 'passed')
headers = rows[0]

# 각 학생 정보를 dict로 변환해서 저장할 리스트이다.
student_list = []

# rows[1:]은 헤더를 제외한 실제 학생 데이터 행만 의미한다.
for row in rows[1:]:
    # zip(headers, row)는 컬럼명과 실제 값을 짝지어 준다.
    # 예: ('name', '홍길동'), ('score', 92)
    # dict()로 감싸면 {'name': '홍길동', 'score': 92} 같은 딕셔너리가 된다.
    student = dict(zip(headers, row))

    # 변환된 학생 dict를 리스트에 추가한다.
    student_list.append(student)

print('headers:', headers)
print('student_list:', student_list)


headers: ('id', 'name', 'course', 'score', 'passed')
student_list: [{'id': 1, 'name': '홍길동', 'course': 'Python IO', 'score': 92, 'passed': True}, {'id': 2, 'name': '이순신', 'course': 'Python IO', 'score': 85, 'passed': True}, {'id': 3, 'name': '유관순', 'course': 'Python IO', 'score': 78, 'passed': True}, {'id': 4, 'name': '신사임당', 'course': 'Python IO', 'score': 64, 'passed': False}]


In [ ]:
# 예제: 읽어온 엑셀 데이터로 간단한 평균 계산
# score 값만 모아 둘 리스트이다.
scores = []

# student_list에는 학생 한 명의 정보가 dict 형태로 들어 있다.
for student in student_list:
    # student['score']는 각 학생의 점수 값이다.
    scores.append(student['score'])

# 평균 = 전체 점수 합계 / 점수 개수
average_score = sum(scores) / len(scores)

print('scores:', scores)
print('average_score:', average_score)


## PDF 파일 읽기

PDF는 페이지, 글자 위치, 폰트, 이미지 정보가 함께 들어 있는 문서 파일이다. 그래서 텍스트 파일처럼 바로 읽기보다 `pypdf` 같은 라이브러리로 페이지 단위 텍스트를 추출한다.

한글 PDF는 폰트와 인코딩 방식에 따라 텍스트 추출 결과가 달라질 수 있으므로, 첫 실습용 PDF는 영문 문장으로 구성했다.


In [7]:
# 예제: PDF 파일을 열고 페이지 수 확인
# pypdf는 PDF 파일을 읽고, 페이지별 텍스트를 추출할 수 있게 해주는 라이브러리이다.
from pypdf import PdfReader

# PdfReader()는 PDF 파일을 열어 Reader 객체로 반환한다.
# 이 객체를 통해 PDF의 페이지 목록에 접근할 수 있다.
reader = PdfReader(pdf_path)

# reader.pages는 PDF의 각 페이지를 담고 있는 리스트와 비슷한 객체이다.
# len(reader.pages)를 사용하면 전체 페이지 수를 확인할 수 있다.
print('len(reader.pages):', len(reader.pages))


len(reader.pages): 2


In [8]:
# 예제: PDF 첫 페이지 텍스트 추출
# reader.pages[0]은 PDF의 첫 번째 페이지를 의미한다.
# Python 리스트 인덱스처럼 0부터 시작한다.
first_page = reader.pages[0]

# extract_text()는 페이지 안에서 추출 가능한 텍스트를 문자열로 반환한다.
# PDF는 문서 구조가 복잡하기 때문에 파일에 따라 추출 결과가 완벽하지 않을 수 있다.
first_page_text = first_page.extract_text()

print('first_page_text:', first_page_text)


first_page_text: IO Practice Report
This PDF is used for Python file I/O practice.
PDF files are structured documents, not plain text files.
We use pypdf to extract text page by page.



In [9]:
# 예제: PDF 전체 페이지 텍스트 추출
# enumerate(reader.pages, start=1)는 페이지 객체와 페이지 번호를 함께 꺼낸다.
# start=1을 사용하면 사용자에게 보여줄 페이지 번호가 1부터 시작한다.
for page_number, page in enumerate(reader.pages, start=1):
    # 현재 페이지에서 텍스트를 추출한다.
    text = page.extract_text()

    # 페이지별 출력 구분선을 만들어 어느 페이지의 내용인지 확인한다.
    print('--- page_number:', page_number, '---')
    print(text)


--- page_number: 1 ---
IO Practice Report
This PDF is used for Python file I/O practice.
PDF files are structured documents, not plain text files.
We use pypdf to extract text page by page.

--- page_number: 2 ---
Second Page
A PDF can contain multiple pages.
In Python, each page can be read and processed separately.



## 수업 정리 포인트

- `.txt`, `.csv`는 비교적 단순한 텍스트 파일이라 `open()` 또는 `csv` 모듈로 다룰 수 있다.
- `.xlsx`, `.pdf`는 내부 구조가 복잡한 문서 파일이라 전용 라이브러리를 사용하는 것이 일반적이다.
- `pandas`를 배우기 전에도 파일 구조에 맞는 라이브러리를 사용하면 데이터를 직접 읽어올 수 있다.
